# 09. TBO InventoryIQ – Feature Engineering & Dataset Transformation
**Phase 3: Production Feature Engineering & Split Manifests**  
**Team:** Team Sabuj Sangha (Abhranil Pal & Arka Dyuti Bose)

This notebook inspects the engineered feature datasets across all travel verticals, verifies quality bounds, reviews the target leakage audit, and examines the chronological train/validation/test split manifests.


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
import json
from feature_engineering.config import FEATURES_DATA_DIR, REPORTS_DIR


## 1. Inspect All 8 Engineered Feature Datasets


In [2]:
summary_path = REPORTS_DIR / "feature_generation_summary.json"
with open(summary_path, "r", encoding="utf-8") as f:
    summary_data = json.load(f)

summary_df = pd.DataFrame(summary_data["datasets"])
summary_df[["dataset_name", "row_count", "column_count", "parquet_size_mb", "csv_size_mb"]]


,dataset_name,row_count,column_count,parquet_size_mb,csv_size_mb
0,hotel_cancellation_features,119208,54,2.12,29.65
1,flight_price_features,300153,21,3.23,44.77
2,flight_operations_features,1000,22,0.04,0.09
3,customer_demand_features,5000,39,0.22,0.93
4,inventory_features,5000,27,0.19,0.80
5,supplier_profitability_features,5000,28,0.45,0.96
6,market_demand_features,5000,26,0.08,0.67
7,destination_demand_supply_features,20,17,0.01,0.00


## 2. Hotel Cancellation Features: Predictive Inputs vs. Analytical References

> **Governance Note on Leakage Quarantine & Model Inputs:**  
> The hotel feature dataset contains 54 columns representing pre-booking signals, operational outcomes, and historical risk benchmarks:
> - **Predictive Inputs (Model-Ready):** Clean Day-0 features available at booking transaction time (e.g., `lead_time_days`, `total_stay_nights`, `adr`, `adr_per_guest`, guest composition, arrival seasonality).
> - **Analytical References Only (Excluded from Model Training):** Features such as `market_segment_historical_cancel_rate` and `deposit_type_historical_cancel_rate` are calculated using Bayesian target averages across the complete dataset. They are preserved for business reporting and historical benchmarking but are **strictly quarantined and excluded from model training inputs** via `TaskConfig.excluded_columns`.
> - **Operational Outcomes (Post-Booking Exclusions):** Attributes assigned at check-in (e.g., `assigned_room_type`, `room_type_changed`, `booking_changes`) are strictly excluded from Day-0 cancellation risk models.


In [3]:
hotel_feats = pd.read_parquet(FEATURES_DATA_DIR / 'hotel' / 'hotel_cancellation_features.parquet')
print(f'Hotel Features Total Shape: {hotel_feats.shape}')

# 1. Predictive Day-0 Inputs (Eligible for Model Training)
predictive_cols = ['lead_time_days', 'lead_time_bucket', 'total_stay_nights', 'adr', 'adr_per_guest']
print('\n--- 1. Predictive Day-0 Inputs (Eligible for Model Training) ---')
display(hotel_feats[predictive_cols].head())

# 2. Analytical Reference Benchmarks (Strictly EXCLUDED from Model Training due to Full-Dataset Target Averaging)
analytical_cols = ['market_segment_historical_cancel_rate', 'deposit_type_historical_cancel_rate']
print('\n--- 2. Analytical Reference Benchmarks (EXCLUDED from Model Training via TaskConfig.excluded_columns) ---')
display(hotel_feats[analytical_cols].head())


Hotel Features Total Shape: (119208, 54)

--- 1. Predictive Day-0 Inputs (Eligible for Model Training) ---

--- 2. Analytical Reference Benchmarks (EXCLUDED from Model Training via TaskConfig.excluded_columns) ---


,lead_time_days,lead_time_bucket,total_stay_nights,adr,adr_per_guest
0,342,181-365d,0,0.0,0.0
1,737,365d+,0,0.0,0.0
2,7,0-7d,1,75.0,75.0
3,13,8-30d,1,75.0,75.0
4,14,8-30d,2,98.0,49.0


,market_segment_historical_cancel_rate,deposit_type_historical_cancel_rate
0,0.1546,0.2841
1,0.1546,0.2841
2,0.1546,0.2841
3,0.1893,0.2841
4,0.3676,0.2841


## 3. Sample Flight Dynamic Pricing Features


In [4]:
price_feats = pd.read_parquet(FEATURES_DATA_DIR / "flight_prices" / "flight_price_features.parquet")
print(f"Flight Price Features Shape: {price_feats.shape}")
price_feats[["price", "days_left", "days_left_bucket", "duration_hours", "duration_bucket", "class", "airline"]].head()


Flight Price Features Shape: (300153, 21)


,price,days_left,days_left_bucket,duration_hours,duration_bucket,class,airline
0,5953.0,1,1-3d,2.17,Short_Haul,Economy,SpiceJet
1,5953.0,1,1-3d,2.33,Short_Haul,Economy,SpiceJet
2,5956.0,1,1-3d,2.17,Short_Haul,Economy,AirAsia
3,5955.0,1,1-3d,2.25,Short_Haul,Economy,Vistara
4,5955.0,1,1-3d,2.33,Short_Haul,Economy,Vistara


## 4. Sample Inventory Utilization & Depletion Velocity Features


In [5]:
inv_feats = pd.read_parquet(FEATURES_DATA_DIR / "inventory" / "inventory_features.parquet")
print(f"Inventory Features Shape: {inv_feats.shape}")
inv_feats[["inventory_id", "travel_date", "snapshot_date", "total_capacity", "occupancy_rate", "availability_rate", "booking_velocity_units_per_day", "inventory_depletion_velocity"]].head()


Inventory Features Shape: (5000, 27)


,inventory_id,travel_date,snapshot_date,total_capacity,occupancy_rate,availability_rate,booking_velocity_units_per_day,inventory_depletion_velocity
0,HOT017,2026-06-29,2026-05-14,110,0.2273,0.7727,0.00,0.00
1,HOT017,2026-06-29,2026-06-09,110,0.4000,0.6000,0.73,0.73
2,HOT017,2026-06-29,2026-06-24,110,0.6182,0.3818,1.60,1.60
3,HOT017,2026-06-29,2026-06-28,110,0.8091,0.1909,5.25,5.25
4,FLT010,2026-10-30,2026-09-15,250,0.2960,0.7040,0.00,0.00


## 5. Sample Supplier Profitability Features


In [6]:
supplier_feats = pd.read_parquet(FEATURES_DATA_DIR / "supplier" / "supplier_profitability_features.parquet")
print(f"Supplier Features Shape: {supplier_feats.shape}")
supplier_feats[["offer_id", "supplier_id", "purchase_cost_inr", "selling_price_inr", "gross_margin_inr", "gross_margin_percentage", "markup_percentage", "is_negative_margin"]].head()


Supplier Features Shape: (5000, 28)


,offer_id,supplier_id,purchase_cost_inr,selling_price_inr,gross_margin_inr,gross_margin_percentage,markup_percentage,is_negative_margin
0,OFF0001,SUP017,21696.08,28874.37,7178.29,24.86,33.09,False
1,OFF0002,SUP019,5437.69,6991.46,1553.77,22.22,28.57,False
2,OFF0003,SUP003,4809.08,6100.82,1291.74,21.17,26.86,False
3,OFF0004,SUP002,4759.89,6153.66,1393.77,22.65,29.28,False
4,OFF0005,SUP003,11505.47,14690.27,3184.80,21.68,27.68,False


## 6. Review Chronological Train/Validation/Test Split Manifests

> **Methodological Note:** For datasets with calendar timestamps (Hotel, Demand, Inventory, Supplier), splits follow strict chronological boundaries. For Flight Prices, the split is an advance booking horizon scenario partition (>=15d advance vs. 8-14d mid-range vs. 1-7d near-departure) assessing model generalization across purchasing urgency.


In [7]:
manifest_path = REPORTS_DIR / "dataset_split_manifest.json"
with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

for ds_name, sinfo in manifest["datasets"].items():
    print(f"Dataset: {ds_name} (Total: {sinfo['total_records']:,} rows, Split Key: {sinfo['split_key']})")
    print(f"  Train: {sinfo['train']['records']:,} ({sinfo['train']['pct']}%)")
    print(f"  Validation: {sinfo['validation']['records']:,} ({sinfo['validation']['pct']}%)")
    print(f"  Test: {sinfo['test']['records']:,} ({sinfo['test']['pct']}%)")


Dataset: hotel_cancellation_features (Total: 119,208 rows, Split Key: arrival_date)
  Train: 78,589 (65.93%)
  Validation: 18,442 (15.47%)
  Test: 22,177 (18.6%)
Dataset: flight_price_features (Total: 300,153 rows, Split Key: days_left)
  Train: 225,235 (75.04%)
  Validation: 42,805 (14.26%)
  Test: 32,113 (10.7%)
Dataset: customer_demand_features (Total: 5,000 rows, Split Key: chronological_index)
  Train: 3,500 (70.0%)
  Validation: 750 (15.0%)
  Test: 750 (15.0%)
Dataset: inventory_features (Total: 5,000 rows, Split Key: chronological_snapshot_date)
  Train: 3,500 (70.0%)
  Validation: 750 (15.0%)
  Test: 750 (15.0%)
Dataset: supplier_profitability_features (Total: 5,000 rows, Split Key: chronological_travel_date)
  Train: 3,500 (70.0%)
  Validation: 750 (15.0%)
  Test: 750 (15.0%)


## 7. Review Feature Dictionary Catalog


In [8]:
dict_df = pd.read_csv(REPORTS_DIR / "feature_dictionary.csv")
print(f"Total Documented Features: {len(dict_df)}")
dict_df[["feature_name", "dataset_name", "data_type", "business_purpose", "leakage_status"]].head(15)


Total Documented Features: 234


,feature_name,dataset_name,data_type,business_purpose,leakage_status
0,is_canceled,hotel_cancellation_features,int64,target_variable,NaN
1,lead_time_days,hotel_cancellation_features,int64,predictive_lead_time,NaN
2,lead_time_bucket,hotel_cancellation_features,object,predictive_risk_tiering,NaN
3,lead_time_log,hotel_cancellation_features,float64,predictive_scale_normalization,NaN
4,booking_date,hotel_cancellation_features,object,metadata_identifier,NaN
5,arrival_year,hotel_cancellation_features,int64,predictive_feature,NaN
6,arrival_month,hotel_cancellation_features,int64,predictive_feature,NaN
7,arrival_day,hotel_cancellation_features,int64,predictive_feature,NaN
8,arrival_day_of_week,hotel_cancellation_features,int64,predictive_feature,NaN
9,arrival_week_of_year,hotel_cancellation_features,int64,predictive_feature,NaN
